# 강의 03 · 실습 1 — 에이전트 동작 원리 · (2-2) 빈칸 채우기 II

## 1. 문제상황

- 구름월드 놀이공원 고객센터에는 운영시간, 주차 요금, 환불 규정을 묻는 질문이 하루 종일 들어옵니다.
- 질문 중에는 「지금 몇 시예요」처럼 FAQ 문서가 아니라 현재 시각을 봐야 답할 수 있는 질문도 섞여 있습니다.
- 담당자는 질문마다 FAQ 문서를 찾아보거나 시계를 본 뒤 답을 씁니다.
- 언어 모델은 구름월드의 FAQ 내용도 현재 시각도 알지 못하므로, 모델에게 질문을 그대로 넘기면 모르는 내용을 지어내거나 답하지 못합니다.

## 2. 문제와 목표

- **문제**: 모델은 구름월드의 FAQ 내용과 현재 시각을 모릅니다. 사람이 질문마다 FAQ 조회와 시각 확인을 대신 해 주어야 합니다.
- **목표**
  - 질문을 입력하면 모델이 두 도구 중 필요한 도구를 스스로 골라 호출해야 합니다.
    - 두 도구: FAQ 조회(`faq_lookup`), 현재 시각(`get_now`)
  - 도구 결과를 받아 최종 답을 만드는 안내 프로그램을 만듭니다.
- **목표 달성 여부의 판정 기준**:
  - 운영시간 질문에서는 `faq_lookup` 도구가, 현재 시각 질문에서는 `get_now` 도구가 호출되고,
  - 두 질문 모두 도구 결과가 반영된 최종 답으로 끝나는 것을 실행 기록에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec03_ex01_s1_diagram.svg)

## 4. 단계별 요구사항

1. **도구를 선언합니다.**
    - FAQ 사전(운영시간·주차·환불)에서 항목을 조회하는 `faq_lookup(topic)`과 현재 날짜·시각을 돌려주는 `get_now()` 두 함수를 만들고, 각 함수 위에 `@tool`을 붙여 도구로 만듭니다.
    - `faq_lookup`은 독스트링에 도구 설명과 `topic` 인자의 설명을 적고, `parse_docstring=True`로 독스트링을 도구 설명(스키마)으로 씁니다.
2. **도구를 모델에 묶습니다.**
    - `bind_tools`로 두 도구를 모델에 붙여 도구를 쥔 모델 `llm_tools`를 만들고, 도구 이름으로 도구 객체를 찾는 사전 `TOOLS`를 만듭니다.
3. **모델을 호출하고 도구 호출 요청을 판정합니다.**
    - 사용자 질문을 `HumanMessage`로 담은 대화 기록을 `llm_tools`에 넣어 호출하고, 돌아온 응답의 `tool_calls`가 비어 있는지로 도구 호출 요청 여부를 판정합니다.
    - `tool_calls`의 각 항목에는 도구 이름, 인자, 호출 id가 들어 있습니다.
4. **도구 결과를 되먹여 반복합니다.**
    - 도구 호출 요청이 있으면 모델 응답을 대화 기록에 붙이고, 요청된 도구를 `TOOLS`에서 찾아 인자로 실행한 뒤, 결과를 호출 id와 함께 `ToolMessage`로 대화 기록에 붙이고 모델을 다시 호출합니다.
    - 도구 호출 요청이 없는 응답이 나오면 그 응답의 내용을 최종 답으로 돌려줍니다.
    - 반복은 4회를 상한으로 하고, 상한에 닿으면 「반복 한도 초과」를 돌려줍니다.
5. **두 질문으로 실행합니다.**
    - 운영시간 질문과 현재 시각 질문을 차례로 넣어, 질문마다 호출된 도구 이름과 인자, 도구 결과, 최종 답을 화면에 출력합니다.

## 5. 코드 골격 — 도구 호출 루프 4단

랭체인 문법으로 도구 호출 루프를 세우는 순서는 다음 네 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 네 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 도구 선언 | 파이썬 함수 위에 표시 한 줄을 붙여 도구로 만듭니다 | `@tool(parse_docstring=True)` | 1 |
| ② 도구 묶기 | 도구 목록을 모델에 붙여 도구를 쥔 모델을 만듭니다 | `llm.bind_tools([...])` | 2 |
| ③ 반복 호출·판정 | 대화 기록을 넣어 호출하고, 도구 호출이 실렸는지 봅니다 | `res.tool_calls` | 3 |
| ④ 결과 되먹임 | 도구를 실행하고 그 결과를 대화 기록에 붙여 다시 호출합니다 | `ToolMessage(content=..., tool_call_id=...)` | 4, 5 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
# 여기에 단계 0(라이브러리 불러오기, .env 읽기, 모델 준비)을 작성합니다.

### 단계 ① — 도구 선언 (요구사항 1)

- 도구의 실체는 파이썬 함수입니다. 함수 위에 `@tool`을 붙이면 함수 이름·독스트링·인자 타입에서 모델에게 보여 줄 도구 설명(스키마)이 만들어집니다.
- `parse_docstring=True`는 독스트링의 `Args:` 항목을 인자 설명으로 씁니다. 모델은 도구 설명과 인자 설명을 보고 어느 도구를 언제 부를지 정합니다.
- `get_now`는 인자가 없으므로 `@tool`만 붙입니다.

In [ ]:
# 여기에 단계 ①(도구 두 개 선언)을 작성합니다.

### 단계 ② — 도구 묶기 (요구사항 2)

- `bind_tools`는 도구 목록을 모델에 붙여, 호출할 때마다 도구 설명을 함께 보내는 새 모델 객체를 돌려줍니다. 원래의 `llm`은 바뀌지 않습니다.
- `TOOLS`는 모델이 보낸 도구 이름을 실제 도구 객체로 바꾸는 사전입니다. 단계 ④에서 도구를 실행할 때 씁니다.

In [ ]:
# 여기에 단계 ②(도구 묶기와 TOOLS 사전)를 작성합니다.

### 단계 ③ — 반복 호출·판정 (요구사항 3)

- 대화 기록은 메시지 객체의 리스트입니다. 첫 항목은 사용자 질문을 담은 `HumanMessage`입니다.
- 도구를 쥔 모델을 호출하면 `AIMessage`가 돌아옵니다. 모델이 도구를 부르기로 정했으면 `tool_calls`에 도구 이름·인자·호출 id가 실립니다. `tool_calls`가 비어 있으면 그 응답이 최종 답입니다.
- 아래 셀은 루프를 돌리기 전에 첫 호출의 응답을 그대로 열어 봅니다. 인자는 처음부터 딕셔너리로 돌아오므로 문자열 파싱이 필요 없습니다.

In [ ]:
# 여기에 단계 ③(첫 호출과 tool_calls 판정)을 작성합니다.

### 단계 ④ — 결과 되먹임 (요구사항 4, 5)

- 도구 호출 요청이 있으면 모델 응답(`AIMessage`)을 먼저 대화 기록에 붙입니다. 그 다음 요청된 도구를 `TOOLS`에서 찾아 `invoke(인자)`로 실행합니다.
- 도구 결과는 `ToolMessage`로 대화 기록에 붙입니다. `tool_call_id`는 어느 요청에 대한 결과인지 모델에게 알려 줍니다.
- 붙인 뒤 모델을 다시 호출합니다. 호출·판정·되먹임을 `for` 문 안에 넣으면 도구 호출 루프가 됩니다. 종료 조건은 도구 호출 요청이 없는 응답과 반복 한도 도달 둘입니다.

In [ ]:
# 여기에 단계 ④(run_agent 함수와 두 질문 실행)를 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 단계 ③의 출력에서 `tool_calls`에 도구 이름 `faq_lookup`, 인자 `{'topic': '운영시간'}`, 호출 id가 실려 있고, 판정이 「도구 호출 요청이 있습니다」로 찍힙니다.
2. 1번 질문(운영시간)에서는 `[도구 호출] faq_lookup {'topic': '운영시간'}`과 `[도구 결과] 매일 09:30~21:00에 운영합니다.`가 찍힌 뒤 최종 답이 나옵니다. 최종 답에 도구 결과의 운영시간이 들어 있습니다.
3. 2번 질문(현재 시각)에서는 `[도구 호출] get_now {}`가 찍힙니다. 최종 답에 도구 결과의 날짜와 시각이 들어 있습니다.

세 가지가 모두 확인되면 완성입니다. 하나라도 다르면 `lec03_ex01_s1.ipynb`와 대조해 채운 빈칸을 고칩니다.